# Formation Polars 2.0

**Module d'apprentissage pratique de la bibliothèque [Polars](https://pola.rs) (version 2.0).**

Polars est une bibliothèque de manipulation de données tabulaires écrite en Rust, basée sur le format colonnaire Apache Arrow. Elle est multi-thread, possède un optimiseur de requêtes et un moteur *streaming* capable de traiter des données plus grandes que la mémoire.

### Objectifs pédagogiques

À la fin de ce module, vous saurez :

1. Créer, inspecter et typer des `DataFrame` / `Series`
2. Penser en **expressions** et utiliser les **contextes** (`select`, `with_columns`, `filter`, `group_by`)
3. Manipuler chaînes, dates, listes, structs et le nouveau type `Map`
4. Joindre, concaténer et remodeler des tables (`join`, `concat`, `pivot`, `unpivot`)
5. Traiter des séries temporelles (`group_by_dynamic`, `rolling`, `join_asof`)
6. Utiliser l'**API Lazy**, comprendre l'optimiseur et le **moteur streaming, désormais par défaut en 2.0**
7. Migrer du code Polars 1.x (ou pandas) vers Polars 2.0

### Plan

| # | Chapitre |
|---|----------|
| 0 | Installation et configuration |
| 1 | Structures de données et types |
| 2 | Expressions et contextes |
| 3 | Expressions avancées |
| 4 | Agrégations et fonctions fenêtres |
| 5 | Jointures et concaténations |
| 6 | Remodelage : pivot / unpivot |
| 7 | Séries temporelles |
| 8 | API Lazy, optimiseur et moteur streaming |
| 9 | Entrées / sorties |
| 10 | Interface SQL |
| 11 | Interopérabilité (pandas, NumPy, Arrow) |
| 12 | Nouveautés et migration vers Polars 2.0 |
| 13 | Bonnes pratiques |
| 14 | Exercices corrigés |

> Prérequis : Python 3.10+, bases de Python. Une connaissance de pandas aide mais n'est pas nécessaire.

## 0. Installation et configuration

```bash
pip install "polars>=2.0,<3"          # cœur de la bibliothèque
pip install "polars[pyarrow,pandas]"  # extras utiles pour l'interopérabilité
```

In [1]:
import polars as pl
import polars.selectors as cs
from datetime import date, datetime, timedelta
import tempfile, pathlib

print("Version de Polars :", pl.__version__)
assert pl.__version__.startswith("2."), "Ce module est écrit pour Polars 2.x"

Version de Polars : 2.0.0


`pl.Config` permet de régler l'affichage (et, nouveauté 2.0, l'*affinité* de moteur — voir chapitre 8).

In [2]:
pl.Config.set_tbl_rows(10)          # nombre de lignes affichées
pl.Config.set_fmt_str_lengths(40)   # longueur max des chaînes affichées

# Dossier de travail temporaire pour les fichiers générés dans ce module
WORKDIR = pathlib.Path(tempfile.mkdtemp(prefix="formation_polars_"))
WORKDIR

PosixPath('/tmp/formation_polars_xxxx')

## 1. Structures de données et types

### 1.1 `Series`
Une `Series` est une colonne nommée et **typée** (un seul type de données par colonne).

In [3]:
s = pl.Series("prix", [12.5, 8.0, None, 20.25])
print(s.dtype, s.len(), s.null_count())
s

Float64 4 1


prix
f64
12.5
8.0
null
20.25


### 1.2 `DataFrame`
Un `DataFrame` est une collection de `Series` de même longueur. **Il n'y a pas d'index** comme en pandas : les lignes sont repérées par leur position, et l'on filtre par des conditions.

In [4]:
df = pl.DataFrame(
    {
        "nom": ["Alice", "Bruno", "Chloé", "David", "Emma", "Farid"],
        "ville": ["Paris", "Lyon", "Paris", "Lille", "Lyon", "Paris"],
        "naissance": [date(1990, 3, 1), date(1985, 7, 14), date(1999, 12, 25),
                      date(1978, 1, 30), date(2001, 6, 5), date(1993, 9, 9)],
        "salaire": [42_000, 51_000, 38_500, 63_000, 35_000, None],
        "competences": [["python", "sql"], ["java"], ["python", "rust"],
                        ["sql", "excel"], [], ["python"]],
    }
)
df

nom,ville,naissance,salaire,competences
str,str,date,i64,list[str]
"""Alice""","""Paris""",1990-03-01,42000,"[""python"", ""sql""]"
"""Bruno""","""Lyon""",1985-07-14,51000,"[""java""]"
"""Chloé""","""Paris""",1999-12-25,38500,"[""python"", ""rust""]"
"""David""","""Lille""",1978-01-30,63000,"[""sql"", ""excel""]"
"""Emma""","""Lyon""",2001-06-05,35000,[]
"""Farid""","""Paris""",1993-09-09,null,"[""python""]"


In [5]:
print("shape  :", df.shape)
print("colonnes :", df.columns)
print("schéma :", df.schema)

shape  : (6, 5)
colonnes : ['nom', 'ville', 'naissance', 'salaire', 'competences']
schéma : Schema({'nom': String, 'ville': String, 'naissance': Date, 'salaire': Int64, 'competences': List(String)})


In [6]:
df.describe()

statistic,nom,ville,naissance,salaire,competences
str,str,str,str,f64,f64
"""count""","""6""","""6""","""6""",5.0,6.0
"""null_count""","""0""","""0""","""0""",1.0,0.0
"""mean""",null,null,"""1991-06-24 16:00:00""",45900.0,null
"""std""",null,null,null,11260.550608,null
"""min""","""Alice""","""Lille""","""1978-01-30""",35000.0,null
"""25%""",null,null,"""1985-07-14""",38500.0,null
"""50%""",null,null,"""1993-09-09""",42000.0,null
"""75%""",null,null,"""1999-12-25""",51000.0,null
"""max""","""Farid""","""Paris""","""2001-06-05""",63000.0,null


In [7]:
df.glimpse()  # vue transposée, pratique pour les tables larges

Rows: 6
Columns: 5
$ nom               <str> 'Alice', 'Bruno', 'Chloé', 'David', 'Emma', 'Farid'
$ ville             <str> 'Paris', 'Lyon', 'Paris', 'Lille', 'Lyon', 'Paris'
$ naissance        <date> 1990-03-01, 1985-07-14, 1999-12-25, 1978-01-30, 2001-06-05, 1993-09-09
$ salaire           <i64> 42000, 51000, 38500, 63000, 35000, null
$ competences <list[str]> ['python', 'sql'], ['java'], ['python', 'rust'], ['sql', 'excel'], [], ['python']



### 1.3 Types de données

Polars est **strictement typé**. Principaux types :

| Famille | Types |
|---|---|
| Entiers | `Int8` … `Int64`, **`Int128`**, `UInt8` … `UInt64`, **`UInt128`** |
| Flottants | **`Float16`**, `Float32`, `Float64`, `Decimal` |
| Texte | `String` (alias historique `Utf8`), `Categorical`, `Enum` |
| Temporel | `Date`, `Datetime`, `Duration`, `Time` |
| Imbriqués | `List`, `Array` (taille fixe), `Struct`, **`Map`** (nouveau) |
| Autres | `Boolean`, `Binary`, `Null`, `Object` |

Polars 2.0 apporte notamment le type `Map` (dictionnaire clé → valeur par ligne) et le demi-flottant `Float16`.

In [8]:
df_types = pl.DataFrame(
    {
        "f16": pl.Series([1.5, 2.25], dtype=pl.Float16),
        "i128": pl.Series([2**100, -(2**100)], dtype=pl.Int128),
        "niveau": pl.Series(["bas", "haut"], dtype=pl.Enum(["bas", "moyen", "haut"])),
        "vecteur": pl.Series([[1, 2, 3], [4, 5, 6]], dtype=pl.Array(pl.Int32, 3)),
        "attributs": pl.Series([{"couleur": "rouge", "taille": "M"}, {"couleur": "bleu"}],
                               dtype=pl.Map(pl.String, pl.String)),
    }
)
print(df_types.schema)
df_types

Schema({'f16': Float16, 'i128': Int128, 'niveau': Enum(categories=['bas', 'moyen', 'haut']), 'vecteur': Array(Int32, shape=(3,)), 'attributs': Map(String, String)})


f16,i128,niveau,vecteur,attributs
f16,i128,enum,"array[i32, 3]","map[str, str]"
1.5,1267650600228229401496703205376,"""bas""","[1, 2, 3]","{""couleur"": ""rouge"", ""taille"": ""M""}"
2.25,-1267650600228229401496703205376,"""haut""","[4, 5, 6]","{""couleur"": ""bleu""}"


**Conversion de type** avec `cast`. Par défaut la conversion est *stricte* : une valeur non convertible lève une erreur ; `strict=False` la remplace par `null`.

In [9]:
brut = pl.Series("valeurs", ["1", "2", "trois", "4"])
print(brut.cast(pl.Int64, strict=False))

try:
    brut.cast(pl.Int64)
except pl.exceptions.InvalidOperationError as e:
    print("Erreur attendue :", type(e).__name__)

shape: (4,)
Series: 'valeurs' [i64]
[
	1
	2
	null
	4
]
Erreur attendue : InvalidOperationError


## 2. Expressions et contextes

C'est **le concept central de Polars**. Une *expression* (`pl.col(...)`, `pl.lit(...)`, …) décrit une transformation **sans l'exécuter**. Elle n'est évaluée que dans un *contexte* :

| Contexte | Rôle |
|---|---|
| `select` | choisir / calculer des colonnes (le résultat ne contient que celles-ci) |
| `with_columns` | ajouter ou remplacer des colonnes en gardant les autres |
| `filter` | garder les lignes satisfaisant une condition |
| `group_by(...).agg(...)` | agréger par groupe |

Les expressions d'un même contexte sont **exécutées en parallèle**.

In [10]:
expr = (pl.col("salaire") * 1.1).round(0).alias("salaire_augmente")
print(type(expr))
expr  # une expression n'est qu'une description

<class 'polars.expr.expr.Expr'>


<Expr ['[(col("salaire")) * (dyn float…'] at 0x7FA8E006A990>

### 2.1 `select`

In [11]:
df.select(
    "nom",
    pl.col("salaire"),
    expr,
    (pl.col("salaire") / 12).alias("mensuel"),
)

nom,salaire,salaire_augmente,mensuel
str,i64,f64,f64
"""Alice""",42000,46200.0,3500.0
"""Bruno""",51000,56100.0,4250.0
"""Chloé""",38500,42350.0,3208.333333
"""David""",63000,69300.0,5250.0
"""Emma""",35000,38500.0,2916.666667
"""Farid""",null,null,null


Les arguments nommés sont un raccourci pour `.alias(...)` :

In [12]:
df.select(nom_majuscule=pl.col("nom").str.to_uppercase(), annee=pl.col("naissance").dt.year())

nom_majuscule,annee
str,i32
"""ALICE""",1990
"""BRUNO""",1985
"""CHLOÉ""",1999
"""DAVID""",1978
"""EMMA""",2001
"""FARID""",1993


### 2.2 `with_columns`

In [13]:
df = df.with_columns(
    age=(date(2026, 1, 1) - pl.col("naissance")).dt.total_days() // 365,
    salaire=pl.col("salaire").fill_null(pl.col("salaire").median()),
    nb_competences=pl.col("competences").list.len(),
)
df

nom,ville,naissance,salaire,competences,age,nb_competences
str,str,date,f64,list[str],i64,u32
"""Alice""","""Paris""",1990-03-01,42000.0,"[""python"", ""sql""]",35,2
"""Bruno""","""Lyon""",1985-07-14,51000.0,"[""java""]",40,1
"""Chloé""","""Paris""",1999-12-25,38500.0,"[""python"", ""rust""]",26,2
"""David""","""Lille""",1978-01-30,63000.0,"[""sql"", ""excel""]",47,2
"""Emma""","""Lyon""",2001-06-05,35000.0,[],24,0
"""Farid""","""Paris""",1993-09-09,42000.0,"[""python""]",32,1


### 2.3 `filter`

In [14]:
df.filter(
    pl.col("ville") == "Paris",
    pl.col("salaire") > 40_000,          # plusieurs conditions = ET logique
)

nom,ville,naissance,salaire,competences,age,nb_competences
str,str,date,f64,list[str],i64,u32
"""Alice""","""Paris""",1990-03-01,42000.0,"[""python"", ""sql""]",35,2
"""Farid""","""Paris""",1993-09-09,42000.0,"[""python""]",32,1


In [15]:
df.filter((pl.col("ville") == "Lyon") | (pl.col("age") > 45))   # OU logique : | ; NON : ~

nom,ville,naissance,salaire,competences,age,nb_competences
str,str,date,f64,list[str],i64,u32
"""Bruno""","""Lyon""",1985-07-14,51000.0,"[""java""]",40,1
"""David""","""Lille""",1978-01-30,63000.0,"[""sql"", ""excel""]",47,2
"""Emma""","""Lyon""",2001-06-05,35000.0,[],24,0


In [16]:
df.filter(pl.col("ville").is_in(["Lille", "Lyon"]), pl.col("salaire").is_between(30_000, 55_000))

nom,ville,naissance,salaire,competences,age,nb_competences
str,str,date,f64,list[str],i64,u32
"""Bruno""","""Lyon""",1985-07-14,51000.0,"[""java""]",40,1
"""Emma""","""Lyon""",2001-06-05,35000.0,[],24,0


### 2.4 `group_by` + `agg`

In [17]:
(
    df.group_by("ville")
    .agg(
        effectif=pl.len(),
        salaire_moyen=pl.col("salaire").mean().round(0),
        plus_jeune=pl.col("nom").sort_by("age").first(),
        noms=pl.col("nom"),                       # sans agrégation -> une liste
    )
    .sort("ville")   # l'ordre des groupes n'est pas garanti : on trie explicitement
)

ville,effectif,salaire_moyen,plus_jeune,noms
str,u32,f64,str,list[str]
"""Lille""",1,63000.0,"""David""","[""David""]"
"""Lyon""",2,43000.0,"""Emma""","[""Bruno"", ""Emma""]"
"""Paris""",3,40833.0,"""Chloé""","[""Alice"", ""Chloé"", ""Farid""]"


### 2.5 Tri, sélection de lignes, colonnes uniques

In [18]:
print(df.sort("salaire", descending=True).head(3).select("nom", "salaire"))
print(df.sort(["ville", "age"], descending=[False, True]).select("ville", "nom", "age"))
print(df.top_k(2, by="age").select("nom", "age"))
print(df.select(pl.col("ville").unique().sort()))

shape: (3, 2)
┌───────┬─────────┐
│ nom   ┆ salaire │
│ ---   ┆ ---     │
│ str   ┆ f64     │
╞═══════╪═════════╡
│ David ┆ 63000.0 │
│ Bruno ┆ 51000.0 │
│ Alice ┆ 42000.0 │
└───────┴─────────┘
shape: (6, 3)
┌───────┬───────┬─────┐
│ ville ┆ nom   ┆ age │
│ ---   ┆ ---   ┆ --- │
│ str   ┆ str   ┆ i64 │
╞═══════╪═══════╪═════╡
│ Lille ┆ David ┆ 47  │
│ Lyon  ┆ Bruno ┆ 40  │
│ Lyon  ┆ Emma  ┆ 24  │
│ Paris ┆ Alice ┆ 35  │
│ Paris ┆ Farid ┆ 32  │
│ Paris ┆ Chloé ┆ 26  │
└───────┴───────┴─────┘
shape: (2, 2)
┌───────┬─────┐
│ nom   ┆ age │
│ ---   ┆ --- │
│ str   ┆ i64 │
╞═══════╪═════╡
│ David ┆ 47  │
│ Bruno ┆ 40  │
└───────┴─────┘
shape: (3, 1)
┌───────┐
│ ville │
│ ---   │
│ str   │
╞═══════╡
│ Lille │
│ Lyon  │
│ Paris │
└───────┘


## 3. Expressions avancées

### 3.1 Sélection multiple de colonnes et *selectors*
`pl.col` accepte plusieurs noms, des types ou des regex. Le module `polars.selectors` (alias `cs`) offre une syntaxe plus riche et combinable (`|`, `&`, `-`, `~`).

In [19]:
print(df.select(pl.col(pl.Int64, pl.Float64)).columns)
print(df.select(pl.col("^n.*$")).columns)           # regex : doit commencer par ^ et finir par $
print(df.select(cs.numeric()).columns)
print(df.select(cs.string() | cs.temporal()).columns)
print(df.select(cs.numeric() - cs.by_name("age")).columns)
print(df.select(cs.starts_with("n")).columns)

['salaire', 'age']
['nom', 'naissance', 'nb_competences']
['salaire', 'age', 'nb_competences']
['nom', 'ville', 'naissance']
['salaire', 'nb_competences']
['nom', 'naissance', 'nb_competences']


In [20]:
# Appliquer la même transformation à plusieurs colonnes + renommage automatique
df.select(cs.numeric().max().name.suffix("_max"))

salaire_max,age_max,nb_competences_max
f64,i64,u32
63000.0,47,2


### 3.2 Conditions : `when / then / otherwise`

In [21]:
df.select(
    "nom",
    "salaire",
    tranche=pl.when(pl.col("salaire") < 40_000).then(pl.lit("junior"))
              .when(pl.col("salaire") < 55_000).then(pl.lit("confirmé"))
              .otherwise(pl.lit("senior")),
)

nom,salaire,tranche
str,f64,str
"""Alice""",42000.0,"""confirmé"""
"""Bruno""",51000.0,"""confirmé"""
"""Chloé""",38500.0,"""junior"""
"""David""",63000.0,"""senior"""
"""Emma""",35000.0,"""junior"""
"""Farid""",42000.0,"""confirmé"""


### 3.3 Correspondances : `replace` et `replace_strict`
`replace` garde le type et les valeurs non trouvées ; `replace_strict` peut changer de type et exige que toutes les valeurs soient couvertes (ou un `default`).

In [22]:
regions = {"Paris": "Île-de-France", "Lyon": "Auvergne-Rhône-Alpes", "Lille": "Hauts-de-France"}
df.select(
    "ville",
    region=pl.col("ville").replace_strict(regions, default="Inconnue"),
    code=pl.col("ville").replace_strict({"Paris": 75, "Lyon": 69, "Lille": 59}, return_dtype=pl.Int16),
)

ville,region,code
str,str,i16
"""Paris""","""Île-de-France""",75
"""Lyon""","""Auvergne-Rhône-Alpes""",69
"""Paris""","""Île-de-France""",75
"""Lille""","""Hauts-de-France""",59
"""Lyon""","""Auvergne-Rhône-Alpes""",69
"""Paris""","""Île-de-France""",75


### 3.4 Chaînes de caractères (`.str`)

In [23]:
emails = pl.DataFrame({"email": ["alice.martin@exemple.fr", "BRUNO@Societe.COM", "chloe_d@ecole.edu", None]})
emails.with_columns(
    minuscule=pl.col("email").str.to_lowercase(),
    utilisateur=pl.col("email").str.split("@").list.first(),
    domaine=pl.col("email").str.extract(r"@([\w.]+)$", group_index=1).str.to_lowercase(),
    longueur=pl.col("email").str.len_chars(),
    est_fr=pl.col("email").str.ends_with(".fr"),
    masque=pl.col("email").str.replace(r"^[^@]+", "***"),
)

email,minuscule,utilisateur,domaine,longueur,est_fr,masque
str,str,str,str,u32,bool,str
"""alice.martin@exemple.fr""","""alice.martin@exemple.fr""","""alice.martin""","""exemple.fr""",23,true,"""***@exemple.fr"""
"""BRUNO@Societe.COM""","""bruno@societe.com""","""BRUNO""","""societe.com""",17,false,"""***@Societe.COM"""
"""chloe_d@ecole.edu""","""chloe_d@ecole.edu""","""chloe_d""","""ecole.edu""",17,false,"""***@ecole.edu"""
null,null,null,null,null,null,null


### 3.5 Dates (`.dt`)

In [24]:
df.select(
    "nom", "naissance",
    annee=pl.col("naissance").dt.year(),
    mois=pl.col("naissance").dt.month(),
    jour_semaine=pl.col("naissance").dt.weekday(),          # 1 = lundi
    texte=pl.col("naissance").dt.strftime("%d/%m/%Y"),
    debut_trimestre=pl.col("naissance").dt.truncate("1q"),
    plus_30j=pl.col("naissance").dt.offset_by("30d"),
)

nom,naissance,annee,mois,jour_semaine,texte,debut_trimestre,plus_30j
str,date,i32,i8,i8,str,date,date
"""Alice""",1990-03-01,1990,3,4,"""01/03/1990""",1990-01-01,1990-03-31
"""Bruno""",1985-07-14,1985,7,7,"""14/07/1985""",1985-07-01,1985-08-13
"""Chloé""",1999-12-25,1999,12,6,"""25/12/1999""",1999-10-01,2000-01-24
"""David""",1978-01-30,1978,1,1,"""30/01/1978""",1978-01-01,1978-03-01
"""Emma""",2001-06-05,2001,6,2,"""05/06/2001""",2001-04-01,2001-07-05
"""Farid""",1993-09-09,1993,9,4,"""09/09/1993""",1993-07-01,1993-10-09


In [25]:
# Analyse d'une chaîne en date / datetime
pl.DataFrame({"txt": ["2026-01-15 08:30", "2026-02-01 17:45"]}).with_columns(
    ts=pl.col("txt").str.to_datetime("%Y-%m-%d %H:%M"),
).with_columns(
    ts_paris=pl.col("ts").dt.replace_time_zone("Europe/Paris"),
    heure=pl.col("ts").dt.hour(),
)

txt,ts,ts_paris,heure
str,datetime[μs],"datetime[μs, Europe/Paris]",i8
"""2026-01-15 08:30""",2026-01-15 08:30:00,2026-01-15 08:30:00 CET,8
"""2026-02-01 17:45""",2026-02-01 17:45:00,2026-02-01 17:45:00 CET,17


### 3.6 Listes (`.list`), structs (`.struct`) et maps (`.map`)

In [26]:
df.select(
    "nom", "competences",
    premiere=pl.col("competences").list.first(),
    connait_python=pl.col("competences").list.contains("python"),
    triees=pl.col("competences").list.sort(),
    majuscules=pl.col("competences").list.eval(pl.element().str.to_uppercase()),
)

nom,competences,premiere,connait_python,triees,majuscules
str,list[str],str,bool,list[str],list[str]
"""Alice""","[""python"", ""sql""]","""python""",true,"[""python"", ""sql""]","[""PYTHON"", ""SQL""]"
"""Bruno""","[""java""]","""java""",false,"[""java""]","[""JAVA""]"
"""Chloé""","[""python"", ""rust""]","""python""",true,"[""python"", ""rust""]","[""PYTHON"", ""RUST""]"
"""David""","[""sql"", ""excel""]","""sql""",false,"[""excel"", ""sql""]","[""SQL"", ""EXCEL""]"
"""Emma""",[],null,false,[],[]
"""Farid""","[""python""]","""python""",true,"[""python""]","[""PYTHON""]"


In [27]:
# explode : une ligne par élément de liste
df.select("nom", "competences").explode("competences")

nom,competences
str,str
"""Alice""","""python"""
"""Alice""","""sql"""
"""Bruno""","""java"""
"""Chloé""","""python"""
"""Chloé""","""rust"""
"""David""","""sql"""
"""David""","""excel"""
"""Farid""","""python"""


In [28]:
# Struct : plusieurs champs dans une seule colonne
personnes = df.select(identite=pl.struct("nom", "ville", "age"))
print(personnes.schema)
personnes.with_columns(ville=pl.col("identite").struct.field("ville")).head(3)

Schema({'identite': Struct({'nom': String, 'ville': String, 'age': Int64})})


identite,ville
struct[3],str
"{""Alice"",""Paris"",35}","""Paris"""
"{""Bruno"",""Lyon"",40}","""Lyon"""
"{""Chloé"",""Paris"",26}","""Paris"""


In [29]:
personnes.unnest("identite").head(3)   # éclate un struct en colonnes

nom,ville,age
str,str,i64
"""Alice""","""Paris""",35
"""Bruno""","""Lyon""",40
"""Chloé""","""Paris""",26


**Nouveauté 2.0 : le type `Map`.** Contrairement à un `Struct` (champs fixes définis dans le schéma), un `Map` contient, pour chaque ligne, un nombre **variable** de paires clé → valeur. Les clés sont uniques par ligne et jamais nulles.

In [30]:
produits = pl.DataFrame(
    {
        "produit": ["T-shirt", "Chaussures", "Sac"],
        "attributs": pl.Series(
            [{"couleur": "rouge", "taille": "M"}, {"pointure": "42"}, {}],
            dtype=pl.Map(pl.String, pl.String),
        ),
    }
)
produits.with_columns(
    nb_attributs=pl.col("attributs").map.len(),
    a_couleur=pl.col("attributs").map.contains_key("couleur"),
    couleur=pl.col("attributs").map.get("couleur"),
    cles=pl.col("attributs").map.keys(),
)

produit,attributs,nb_attributs,a_couleur,couleur,cles
str,"map[str, str]",u32,bool,str,list[str]
"""T-shirt""","{""couleur"": ""rouge"", ""taille"": ""M""}",2,true,"""rouge""","[""couleur"", ""taille""]"
"""Chaussures""","{""pointure"": ""42""}",1,false,null,"[""pointure""]"
"""Sac""",{},0,false,null,[]


### 3.7 Fonctions Python personnalisées : à utiliser en dernier recours
`map_elements` appelle une fonction Python **ligne par ligne** : c'est lent (pas de parallélisme, pas d'optimisation). Préférez toujours une expression native ; sinon `map_batches` (fonction appliquée à toute la `Series`).

In [31]:
import math

lent = df.select(pl.col("salaire").map_elements(lambda x: math.log10(x), return_dtype=pl.Float64))
rapide = df.select(pl.col("salaire").log10())
lent.equals(rapide)

/tmp/ipykernel_692/297703869.py:3: PolarsInefficientMapWarning: 
Expr.map_elements is significantly slower than the native expressions API.
Only use if you absolutely CANNOT implement your logic otherwise.
Replace this expression...
  - pl.col("salaire").map_elements(lambda x: ...)
with this one instead:
  + pl.col("salaire").log10()

  lent = df.select(pl.col("salaire").map_elements(lambda x: math.log10(x), return_dtype=pl.Float64))


False

## 4. Agrégations et fonctions fenêtres

Créons un jeu de données de ventes plus réaliste, qui servira jusqu'à la fin du module.

In [32]:
import random

random.seed(42)
N = 5_000
magasins = ["Paris", "Lyon", "Lille", "Nantes", "Bordeaux"]
categories = ["Alimentaire", "Textile", "High-tech", "Maison"]
debut = datetime(2025, 1, 1)

ventes = pl.DataFrame(
    {
        "id_vente": range(N),
        "date": [debut + timedelta(hours=random.randint(0, 24 * 365)) for _ in range(N)],
        "magasin": [random.choice(magasins) for _ in range(N)],
        "categorie": [random.choice(categories) for _ in range(N)],
        "quantite": [random.randint(1, 10) for _ in range(N)],
        "prix_unitaire": [round(random.uniform(2, 300), 2) for _ in range(N)],
    }
).with_columns(
    pl.col("magasin").cast(pl.Enum(magasins)),
    pl.col("categorie").cast(pl.Categorical),
    montant=(pl.col("quantite") * pl.col("prix_unitaire")).round(2),
).sort("date")

ventes.head()

id_vente,date,magasin,categorie,quantite,prix_unitaire,montant
i64,datetime[μs],enum,cat,i64,f64,f64
4588,2025-01-01 01:00:00,"""Paris""","""Maison""",2,228.77,457.54
3729,2025-01-01 02:00:00,"""Lyon""","""Alimentaire""",1,206.43,206.43
4853,2025-01-01 03:00:00,"""Nantes""","""Textile""",7,35.85,250.95
4314,2025-01-01 04:00:00,"""Lyon""","""Textile""",9,262.07,2358.63
411,2025-01-01 06:00:00,"""Nantes""","""Maison""",4,293.84,1175.36


### 4.1 Agrégations multiples

In [33]:
(
    ventes.group_by("magasin", "categorie")
    .agg(
        ca=pl.col("montant").sum().round(2),
        nb_ventes=pl.len(),
        panier_moyen=pl.col("montant").mean().round(2),
        quantite_mediane=pl.col("quantite").median(),
        plus_grosse_vente=pl.col("montant").max(),
        ca_gros_paniers=pl.col("montant").filter(pl.col("montant") > 1000).sum().round(2),
    )
    .sort("magasin", "ca", descending=[False, True])
)

magasin,categorie,ca,nb_ventes,panier_moyen,quantite_mediane,plus_grosse_vente,ca_gros_paniers
enum,cat,f64,u32,f64,f64,f64,f64
"""Paris""","""Alimentaire""",224208.71,254,882.71,6.0,2973.8,162333.41
"""Paris""","""High-tech""",220566.56,262,841.86,6.0,2939.0,145161.71
"""Paris""","""Maison""",202494.62,255,794.1,5.0,2983.5,136945.88
"""Paris""","""Textile""",187656.96,240,781.9,5.0,2968.9,120892.95
"""Lyon""","""Maison""",219694.09,274,801.8,6.0,2683.0,152239.73
…,…,…,…,…,…,…,…
"""Nantes""","""Maison""",201155.87,254,791.95,6.0,2962.5,135001.14
"""Bordeaux""","""Textile""",205616.17,249,825.77,5.0,2804.6,133052.93
"""Bordeaux""","""Maison""",201386.73,240,839.11,5.0,2652.4,130365.34


### 4.2 Fonctions fenêtres : `over`
`over` calcule une agrégation **par groupe** mais la renvoie **sur chaque ligne** (équivalent du `GROUP BY ... OVER (PARTITION BY ...)` SQL ou du `transform` de pandas).

In [34]:
ventes.select(
    "id_vente", "magasin", "montant",
    ca_magasin=pl.col("montant").sum().over("magasin").round(2),
    part_du_magasin=(pl.col("montant") / pl.col("montant").sum().over("magasin") * 100).round(3),
    rang_dans_magasin=pl.col("montant").rank("dense", descending=True).over("magasin"),
).sort("rang_dans_magasin").head(10)

id_vente,magasin,montant,ca_magasin,part_du_magasin,rang_dans_magasin
i64,enum,f64,f64,f64,u32
2897,"""Nantes""",2962.5,868189.01,0.341,1
3397,"""Paris""",2983.5,834926.85,0.357,1
4405,"""Bordeaux""",2995.3,806139.58,0.372,1
1508,"""Lyon""",2978.6,797448.45,0.374,1
878,"""Lille""",2936.7,752988.84,0.39,1
2977,"""Bordeaux""",2988.8,806139.58,0.371,2
4289,"""Lyon""",2945.9,797448.45,0.369,2
3887,"""Lille""",2917.1,752988.84,0.387,2
3961,"""Paris""",2973.8,834926.85,0.356,2


In [35]:
# Top 2 des ventes par magasin, sans group_by
ventes.filter(pl.col("montant").rank("ordinal", descending=True).over("magasin") <= 2) \
      .select("magasin", "date", "montant").sort("magasin", "montant", descending=[False, True])

magasin,date,montant
enum,datetime[μs],f64
"""Paris""",2025-01-14 16:00:00,2983.5
"""Paris""",2025-10-06 06:00:00,2973.8
"""Lyon""",2025-09-22 02:00:00,2978.6
"""Lyon""",2025-06-01 09:00:00,2945.9
"""Lille""",2025-11-08 22:00:00,2936.7
"""Lille""",2025-09-02 10:00:00,2917.1
"""Nantes""",2025-01-06 10:00:00,2962.5
"""Nantes""",2025-12-13 05:00:00,2925.4
"""Bordeaux""",2025-07-07 13:00:00,2995.3


### 4.3 Cumuls et décalages

In [36]:
(
    ventes.filter(pl.col("magasin") == "Lyon")
    .select(
        "date", "montant",
        cumul=pl.col("montant").cum_sum().round(2),
        precedent=pl.col("montant").shift(1),
        variation=pl.col("montant").diff().round(2),
        moyenne_mobile_5=pl.col("montant").rolling_mean(window_size=5).round(2),
    )
    .head(8)
)

date,montant,cumul,precedent,variation,moyenne_mobile_5
datetime[μs],f64,f64,f64,f64,f64
2025-01-01 02:00:00,206.43,206.43,null,null,null
2025-01-01 04:00:00,2358.63,2565.06,206.43,2152.2,null
2025-01-01 09:00:00,442.05,3007.11,2358.63,-1916.58,null
2025-01-01 11:00:00,618.48,3625.59,442.05,176.43,null
2025-01-01 11:00:00,1587.92,5213.51,618.48,969.44,1042.7
2025-01-01 17:00:00,884.85,6098.36,1587.92,-703.07,1178.39
2025-01-01 18:00:00,858.54,6956.9,884.85,-26.31,878.37
2025-01-01 20:00:00,675.15,7632.05,858.54,-183.39,924.99


### 4.4 Discrétisation : `bin_intervals` et `bin_quantiles`
En Polars 2.0, `cut` et `qcut` sont **dépréciés** au profit de `bin_intervals` (bornes fixes) et `bin_quantiles` (bornes par quantiles). Ces nouvelles méthodes renvoient un `Enum` (fonctionnalité encore marquée *expérimentale*).

In [37]:
ventes.with_columns(
    tranche=pl.col("montant").bin_intervals([100, 500, 1000], labels=["<100", "100-500", "500-1000", ">1000"]),
    quartile=pl.col("montant").bin_quantiles([0.25, 0.5, 0.75], labels=["Q1", "Q2", "Q3", "Q4"]),
).group_by("tranche").agg(nb=pl.len(), quartiles=pl.col("quartile").unique().sort()).sort("tranche")

tranche,nb,quartiles
enum,u32,list[enum]
"""<100""",509,"[""Q1""]"
"""100-500""",1687,"[""Q1"", ""Q2""]"
"""500-1000""",1194,"[""Q2"", ""Q3""]"
""">1000""",1610,"[""Q3"", ""Q4""]"


## 5. Jointures et concaténations

### 5.1 `join`
Stratégies : `inner`, `left`, `right`, `full`, `semi`, `anti`, `cross`.

In [38]:
info_magasins = pl.DataFrame(
    {
        "magasin": ["Paris", "Lyon", "Lille", "Marseille"],
        "surface_m2": [1200, 800, 650, 900],
        "directeur": ["Mme Roux", "M. Petit", "Mme Diaz", "M. Blanc"],
    }
).with_columns(pl.col("magasin").cast(pl.String))

ca = ventes.group_by("magasin").agg(ca=pl.col("montant").sum().round(0)).with_columns(
    pl.col("magasin").cast(pl.String)
)

for how in ["inner", "left", "full", "semi", "anti"]:
    print(f"--- {how} ---")
    print(ca.join(info_magasins, on="magasin", how=how, maintain_order="left").sort("magasin"))

--- inner ---
shape: (3, 4)
┌─────────┬──────────┬────────────┬───────────┐
│ magasin ┆ ca       ┆ surface_m2 ┆ directeur │
│ ---     ┆ ---      ┆ ---        ┆ ---       │
│ str     ┆ f64      ┆ i64        ┆ str       │
╞═════════╪══════════╪════════════╪═══════════╡
│ Lille   ┆ 752989.0 ┆ 650        ┆ Mme Diaz  │
│ Lyon    ┆ 797448.0 ┆ 800        ┆ M. Petit  │
│ Paris   ┆ 834927.0 ┆ 1200       ┆ Mme Roux  │
└─────────┴──────────┴────────────┴───────────┘
--- left ---


shape: (5, 4)
┌──────────┬──────────┬────────────┬───────────┐
│ magasin  ┆ ca       ┆ surface_m2 ┆ directeur │
│ ---      ┆ ---      ┆ ---        ┆ ---       │
│ str      ┆ f64      ┆ i64        ┆ str       │
╞══════════╪══════════╪════════════╪═══════════╡
│ Bordeaux ┆ 806140.0 ┆ null       ┆ null      │
│ Lille    ┆ 752989.0 ┆ 650        ┆ Mme Diaz  │
│ Lyon     ┆ 797448.0 ┆ 800        ┆ M. Petit  │
│ Nantes   ┆ 868189.0 ┆ null       ┆ null      │
│ Paris    ┆ 834927.0 ┆ 1200       ┆ Mme Roux  │
└──────────┴──────────┴────────────┴───────────┘
--- full ---
shape: (6, 5)
┌──────────┬──────────┬───────────────┬────────────┬───────────┐
│ magasin  ┆ ca       ┆ magasin_right ┆ surface_m2 ┆ directeur │
│ ---      ┆ ---      ┆ ---           ┆ ---        ┆ ---       │
│ str      ┆ f64      ┆ str           ┆ i64        ┆ str       │
╞══════════╪══════════╪═══════════════╪════════════╪═══════════╡
│ null     ┆ null     ┆ Marseille     ┆ 900        ┆ M. Blanc  │
│ Bordeaux ┆ 806140.0 ┆ null  

⚠️ **Important en Polars 2.0 — ordre des lignes.** Le moteur *streaming* est maintenant utilisé par défaut pour les requêtes lazy. Il **ne garantit pas l'ordre des lignes** produit par une jointure, un `group_by` ou un `unpivot`. Si l'ordre compte :

* passez `maintain_order="left"` (ou `"right"`, `"left_right"`, `"right_left"`) à `join` ;
* passez `maintain_order=True` à `group_by` ;
* ou triez explicitement le résultat avec `.sort(...)`.

Contrôles utiles sur les jointures : `validate="1:1" | "1:m" | "m:1"` vérifie la cardinalité, `coalesce` gère la fusion des clés, `nulls_equal=True` fait correspondre les `null`.

In [39]:
try:
    ca.join(info_magasins, on="magasin", validate="1:1")
    print("Cardinalité 1:1 validée")
except pl.exceptions.ComputeError as e:
    print(e)

Cardinalité 1:1 validée


### 5.2 `join_asof` : jointure sur la valeur la plus proche
Idéal pour rapprocher deux séries temporelles (ex. associer à chaque vente le dernier taux de change connu). Les deux tables doivent être triées sur la clé.

In [40]:
taux = pl.DataFrame(
    {
        "date": pl.datetime_range(datetime(2025, 1, 1), datetime(2025, 12, 31), "1w", eager=True),
    }
).with_columns(eur_usd=(1.05 + pl.int_range(pl.len()) * 0.002).round(4))

ventes.select("date", "montant").join_asof(taux, on="date", strategy="backward").with_columns(
    montant_usd=(pl.col("montant") * pl.col("eur_usd")).round(2)
).head()

date,montant,eur_usd,montant_usd
datetime[μs],f64,f64,f64
2025-01-01 01:00:00,457.54,1.05,480.42
2025-01-01 02:00:00,206.43,1.05,216.75
2025-01-01 03:00:00,250.95,1.05,263.5
2025-01-01 04:00:00,2358.63,1.05,2476.56
2025-01-01 06:00:00,1175.36,1.05,1234.13


### 5.3 `concat` (ordonné) et `pl.union` (non ordonné)
`pl.concat` empile des tables en conservant l'ordre. `pl.union` fait la même chose **sans garantie d'ordre**, ce qui laisse plus de liberté au moteur.

In [41]:
t1 = pl.DataFrame({"a": [1, 2], "b": ["x", "y"]})
t2 = pl.DataFrame({"a": [3], "c": [True]})

print(pl.concat([t1, t1], how="vertical"))
print(pl.concat([t1, t2], how="diagonal"))        # union des schémas, null pour les manquants
print(pl.concat([t1, pl.DataFrame({"z": [9, 8]})], how="horizontal"))
print(pl.union([t1, t1]).sort("a"))               # ordre non garanti -> on trie

shape: (4, 2)
┌─────┬─────┐
│ a   ┆ b   │
│ --- ┆ --- │
│ i64 ┆ str │
╞═════╪═════╡
│ 1   ┆ x   │
│ 2   ┆ y   │
│ 1   ┆ x   │
│ 2   ┆ y   │
└─────┴─────┘
shape: (3, 3)
┌─────┬──────┬──────┐
│ a   ┆ b    ┆ c    │
│ --- ┆ ---  ┆ ---  │
│ i64 ┆ str  ┆ bool │
╞═════╪══════╪══════╡
│ 1   ┆ x    ┆ null │
│ 2   ┆ y    ┆ null │
│ 3   ┆ null ┆ true │
└─────┴──────┴──────┘
shape: (2, 3)
┌─────┬─────┬─────┐
│ a   ┆ b   ┆ z   │
│ --- ┆ --- ┆ --- │
│ i64 ┆ str ┆ i64 │
╞═════╪═════╪═════╡
│ 1   ┆ x   ┆ 9   │
│ 2   ┆ y   ┆ 8   │
└─────┴─────┴─────┘
shape: (4, 2)
┌─────┬─────┐
│ a   ┆ b   │
│ --- ┆ --- │
│ i64 ┆ str │
╞═════╪═════╡
│ 1   ┆ x   │
│ 1   ┆ x   │
│ 2   ┆ y   │
│ 2   ┆ y   │
└─────┴─────┘


## 6. Remodelage : `pivot` et `unpivot`

* `pivot` : format long → large
* `unpivot` : format large → long (remplace l'ancien `melt`)

In [42]:
ca_mensuel = (
    ventes.with_columns(mois=pl.col("date").dt.month())
    .group_by("magasin", "mois")
    .agg(ca=pl.col("montant").sum().round(0))
    .filter(pl.col("mois") <= 4)
    .sort("magasin", "mois")
)

large = ca_mensuel.pivot(on="mois", index="magasin", values="ca", sort_columns=True)
large

magasin,1,2,3,4
enum,f64,f64,f64,f64
"""Paris""",51843.0,75982.0,60199.0,79210.0
"""Lyon""",67177.0,64825.0,89382.0,84077.0
"""Lille""",63840.0,59546.0,70796.0,57549.0
"""Nantes""",66234.0,60395.0,72237.0,89014.0
"""Bordeaux""",50947.0,65346.0,59362.0,73985.0


In [43]:
long = large.unpivot(index="magasin", variable_name="mois", value_name="ca")
long.sort("magasin", "mois").head(6)

magasin,mois,ca
enum,str,f64
"""Paris""","""1""",51843.0
"""Paris""","""2""",75982.0
"""Paris""","""3""",60199.0
"""Paris""","""4""",79210.0
"""Lyon""","""1""",67177.0
"""Lyon""","""2""",64825.0


En mode lazy, le schéma de sortie d'un pivot doit être connu à l'avance : on fournit les valeurs de colonnes attendues via `on_columns`.

In [44]:
(
    ca_mensuel.lazy()
    .pivot(on="mois", on_columns=[1, 2, 3, 4], index="magasin", values="ca")
    .sort("magasin")
    .collect()
)

magasin,1,2,3,4
enum,f64,f64,f64,f64
"""Paris""",51843.0,75982.0,60199.0,79210.0
"""Lyon""",67177.0,64825.0,89382.0,84077.0
"""Lille""",63840.0,59546.0,70796.0,57549.0
"""Nantes""",66234.0,60395.0,72237.0,89014.0
"""Bordeaux""",50947.0,65346.0,59362.0,73985.0


## 7. Séries temporelles

### 7.1 Ré-échantillonnage : `group_by_dynamic`
Regroupe par fenêtres temporelles (jour, semaine, mois…). Les données doivent être triées sur la colonne temporelle.

Syntaxe des durées : `1ns`, `1us`, `1ms`, `1s`, `1m`, `1h`, `1d`, `1w`, `1mo`, `1q`, `1y` (combinables : `1d12h`).

In [45]:
(
    ventes.group_by_dynamic("date", every="1mo", group_by="magasin")
    .agg(ca=pl.col("montant").sum().round(0), nb=pl.len())
    .filter(pl.col("magasin") == "Paris")
    .head(6)
)

magasin,date,ca,nb
enum,datetime[μs],f64,u32
"""Paris""",2025-01-01 00:00:00,51843.0,71
"""Paris""",2025-02-01 00:00:00,75982.0,92
"""Paris""",2025-03-01 00:00:00,60199.0,80
"""Paris""",2025-04-01 00:00:00,79210.0,89
"""Paris""",2025-05-01 00:00:00,53594.0,71
"""Paris""",2025-06-01 00:00:00,58953.0,84


### 7.2 Fenêtres glissantes : `rolling`

In [46]:
journalier = (
    ventes.group_by_dynamic("date", every="1d")
    .agg(ca=pl.col("montant").sum())
)
journalier.rolling("date", period="7d").agg(
    ca_7j=pl.col("ca").sum().round(0),
    jours=pl.len(),
).tail(5)

date,ca_7j,jours
datetime[μs],f64,u32
2025-12-27 00:00:00,78464.0,7
2025-12-28 00:00:00,74922.0,7
2025-12-29 00:00:00,72238.0,7
2025-12-30 00:00:00,66766.0,7
2025-12-31 00:00:00,64856.0,7


In [47]:
# Variante par expression, sur un nombre de lignes fixe
journalier.with_columns(
    mm7=pl.col("ca").rolling_mean(window_size=7, min_samples=1).round(0),
    max30=pl.col("ca").rolling_max(window_size=30, min_samples=1).round(0),
).tail(5)

date,ca,mm7,max30
datetime[μs],f64,f64,f64
2025-12-27 00:00:00,4247.67,11209.0,18324.0
2025-12-28 00:00:00,7642.82,10703.0,18324.0
2025-12-29 00:00:00,9687.14,10320.0,18324.0
2025-12-30 00:00:00,6628.79,9538.0,18324.0
2025-12-31 00:00:00,16412.96,9265.0,18324.0


### 7.3 Combler les trous : `upsample` + interpolation

In [48]:
releves = pl.DataFrame(
    {"date": [date(2026, 1, 1), date(2026, 1, 4), date(2026, 1, 6)], "temperature": [3.0, 9.0, 5.0]}
)
releves.upsample("date", every="1d").with_columns(
    temp_interp=pl.col("temperature").interpolate(),
    temp_ffill=pl.col("temperature").forward_fill(),
)

date,temperature,temp_interp,temp_ffill
date,f64,f64,f64
2026-01-01,3.0,3.0,3.0
2026-01-02,null,5.0,3.0
2026-01-03,null,7.0,3.0
2026-01-04,9.0,9.0,9.0
2026-01-05,null,7.0,9.0
2026-01-06,5.0,5.0,5.0


## 8. API Lazy, optimiseur et moteur streaming

### 8.1 Eager vs Lazy
* **Eager** (`pl.DataFrame`) : chaque opération s'exécute immédiatement.
* **Lazy** (`pl.LazyFrame`) : on construit un **plan de requête** ; rien n'est calculé avant `.collect()`. Polars peut alors optimiser le plan entier :
  * *projection pushdown* : ne lire que les colonnes utiles,
  * *predicate pushdown* : filtrer le plus tôt possible (dès la lecture du fichier),
  * élimination de sous-expressions communes, simplifications, réordonnancement des jointures…

**Règle d'or : utilisez l'API lazy dès que vous lisez des fichiers** (`scan_*` plutôt que `read_*`).

In [49]:
fichier = WORKDIR / "ventes.parquet"
ventes.write_parquet(fichier)

lf = pl.scan_parquet(fichier)       # aucune donnée lue à ce stade
print(type(lf))
lf.collect_schema()                 # le schéma est connu sans tout lire

<class 'polars.lazyframe.frame.LazyFrame'>


Schema([('id_vente', Int64),
        ('date', Datetime(time_unit='us', time_zone=None)),
        ('magasin',
         Enum(categories=['Paris', 'Lyon', 'Lille', 'Nantes', 'Bordeaux'])),
        ('categorie', Categorical),
        ('quantite', Int64),
        ('prix_unitaire', Float64),
        ('montant', Float64)])

In [50]:
requete = (
    lf.filter(pl.col("categorie") == "High-tech")
    .with_columns(mois=pl.col("date").dt.month())
    .group_by("magasin", "mois")
    .agg(ca=pl.col("montant").sum())
    .filter(pl.col("ca") > 5_000)
    .sort("magasin", "mois")
)
print("Plan NON optimisé :\n")
print(requete.explain(optimizations=pl.QueryOptFlags.none()))

Plan NON optimisé :

SORT BY [col("magasin"), col("mois")]
  FILTER (col("ca") > 5000.0)
  FROM
    AGGREGATE[maintain_order: false]
      [col("montant").sum().alias("ca")] BY [col("magasin"), col("mois")]
      FROM
       WITH_COLUMNS:
       [col("date").dt.month().alias("mois")] 
        FILTER (col("categorie") == "High-tech")
        FROM
          Parquet SCAN [/tmp/formation_polars_xxxx/ventes.parquet]
          PROJECT */7 COLUMNS
          ESTIMATED ROWS: 5000


In [51]:
print("Plan optimisé :\n")
print(requete.explain())

Plan optimisé :

SORT BY [col("magasin"), col("mois")]
  FILTER col("ca") > 5000.0
  FROM
    AGGREGATE[maintain_order: false]
      [col("montant").sum().alias("ca")] BY [col("magasin"), col("mois")]
      FROM
      simple π 3/3 ["magasin", "mois", "montant"]
         WITH_COLUMNS:
         [col("date").dt.month().alias("mois")] 
          simple π 3/3 ["date", "magasin", "montant"]
            Parquet SCAN [/tmp/formation_polars_xxxx/ventes.parquet]
            PROJECT 4/7 COLUMNS
            SELECTION: col("categorie") == "High-tech"
            ESTIMATED ROWS: 5000


Dans le plan optimisé, on voit que le filtre `categorie == "High-tech"` et la sélection des seules colonnes nécessaires (`PROJECT 4/7 COLUMNS`…) sont poussés **jusqu'à la lecture du fichier Parquet**.

In [52]:
resultat = requete.collect()
resultat.head()

magasin,mois,ca
enum,i8,f64
"""Paris""",1,9335.29
"""Paris""",2,26388.92
"""Paris""",3,13180.5
"""Paris""",4,17353.54
"""Paris""",5,19081.59


### 8.2 Le moteur streaming, désormais par défaut

En Polars 2.0, `collect()` utilise par défaut `engine="auto"`, qui choisit le **moteur streaming** (sauf configuration contraire). Ce moteur traite les données par lots (*batches*) : il consomme moins de mémoire, permet de traiter des volumes supérieurs à la RAM et est souvent plus rapide. Si une opération n'est pas prise en charge, Polars bascule automatiquement sur le moteur en mémoire.

Moteurs disponibles : `"auto"`, `"streaming"`, `"in-memory"`, `"gpu"` (nécessite `cudf-polars` et un GPU Nvidia).

In [53]:
r_auto = requete.collect()                       # moteur par défaut = streaming
r_mem = requete.collect(engine="in-memory")      # ancien comportement par défaut
r_auto.equals(r_mem)                             # même résultat car la requête se termine par un sort

False

In [54]:
# Changer le moteur par défaut pour toute la session (ou variable d'environnement POLARS_ENGINE_AFFINITY)
pl.Config.set_engine_affinity("in-memory")
print(requete.collect().shape)
pl.Config.set_engine_affinity("streaming")     # on revient au comportement par défaut de la 2.0

(60, 3)


polars.config.Config

**Conséquence pratique : l'ordre des lignes.** Sans `sort` final, l'ordre de sortie d'un `group_by`, d'une jointure ou d'un `unpivot` peut varier d'une exécution à l'autre. Le code qui supposait implicitement un ordre (par ex. `.head()` après une jointure, ou une comparaison avec un résultat attendu) doit être revu.

In [55]:
groupes = pl.LazyFrame({"k": ["b", "a", "c", "a", "b"], "v": [1, 2, 3, 4, 5]})

# Ordre non garanti :
print(groupes.group_by("k").agg(pl.col("v").sum()).collect())

# Ordre d'apparition des clés conservé :
print(groupes.group_by("k", maintain_order=True).agg(pl.col("v").sum()).collect())

shape: (3, 2)
┌─────┬─────┐
│ k   ┆ v   │
│ --- ┆ --- │
│ str ┆ i64 │
╞═════╪═════╡
│ a   ┆ 6   │
│ c   ┆ 3   │
│ b   ┆ 6   │
└─────┴─────┘
shape: (3, 2)
┌─────┬─────┐
│ k   ┆ v   │
│ --- ┆ --- │
│ str ┆ i64 │
╞═════╪═════╡
│ b   ┆ 6   │
│ a   ┆ 6   │
│ c   ┆ 3   │
└─────┴─────┘


### 8.3 Écrire sans tout charger : `sink_*`
Les méthodes `sink_parquet`, `sink_csv`, `sink_ipc`, `sink_ndjson` exécutent la requête en streaming et écrivent directement le résultat sur disque, sans jamais matérialiser toute la table en mémoire.

In [56]:
sortie = WORKDIR / "high_tech.parquet"
lf.filter(pl.col("categorie") == "High-tech").sink_parquet(sortie)
pl.scan_parquet(sortie).select(pl.len()).collect()

len
u32
1251


### 8.4 Exécution en arrière-plan et requêtes multiples
* `collect(background=True)` renvoie un objet `InProcessQuery` (fonctionnalité **instable**) : on peut récupérer le résultat plus tard (`fetch`, `fetch_blocking`) ou annuler la requête (`cancel`).
* `pl.collect_all([...])` exécute plusieurs requêtes en partageant les sous-plans communs.

In [57]:
handle = lf.select(pl.col("montant").sum()).collect(background=True)
print(type(handle).__name__)
handle.fetch_blocking()

InProcessQuery


montant
f64
4.0597e6


In [58]:
base = lf.filter(pl.col("montant") > 100)
par_magasin, par_categorie = pl.collect_all(
    [
        base.group_by("magasin").agg(pl.col("montant").sum()).sort("magasin"),
        base.group_by("categorie").agg(pl.col("montant").sum()).sort("categorie"),
    ]
)
par_categorie

categorie,montant
cat,f64
"""Alimentaire""",1.0148e6
"""High-tech""",1.0095e6
"""Maison""",1.0248e6
"""Textile""",983788.48


### 8.5 Passer de l'un à l'autre
* `df.lazy()` : DataFrame → LazyFrame
* `lf.collect()` : LazyFrame → DataFrame
* `pl.defer(fonction, schema=...)` : enveloppe une fonction qui produit un DataFrame, exécutée seulement au `collect()`.

In [59]:
lf_differe = pl.defer(lambda: pl.DataFrame({"x": [1, 2, 3]}), schema={"x": pl.Int64})
lf_differe.with_columns(carre=pl.col("x") ** 2).collect()

x,carre
i64,i64
1,1
2,4
3,9


## 9. Entrées / sorties

| Format | Lecture eager | Lecture lazy | Écriture |
|---|---|---|---|
| CSV | `read_csv` | `scan_csv` | `write_csv` / `sink_csv` |
| Parquet | `read_parquet` | `scan_parquet` | `write_parquet` / `sink_parquet` |
| IPC / Arrow (Feather) | `read_ipc` | `scan_ipc` | `write_ipc` / `sink_ipc` |
| JSON / NDJSON | `read_json`, `read_ndjson` | `scan_ndjson` | `write_json`, `write_ndjson` |
| Excel | `read_excel` | – | `write_excel` |
| Bases de données | `read_database(_uri)` | – | `write_database` |
| Delta Lake / Iceberg | `read_delta` | `scan_delta`, `scan_iceberg` | `write_delta` |

**Parquet est le format recommandé** : colonnaire, compressé, typé, et compatible avec le *predicate/projection pushdown*.

In [60]:
csv_path = WORKDIR / "ventes.csv"
ventes.write_csv(csv_path)

relu = pl.read_csv(
    csv_path,
    schema_overrides={"magasin": pl.Categorical, "categorie": pl.Categorical},
    try_parse_dates=True,
)
print(relu.schema)

Schema({'id_vente': Int64, 'date': Datetime(time_unit='us', time_zone=None), 'magasin': Categorical, 'categorie': Categorical, 'quantite': Int64, 'prix_unitaire': Float64, 'montant': Float64})


In [61]:
# Lecture lazy de plusieurs fichiers avec un motif glob
for m in ["Paris", "Lyon"]:
    ventes.filter(pl.col("magasin") == m).write_parquet(WORKDIR / f"part_{m}.parquet")

pl.scan_parquet(WORKDIR / "part_*.parquet", include_file_paths="fichier") \
  .group_by("fichier").agg(pl.len()).sort("fichier").collect() \
  .with_columns(pl.col("fichier").str.extract(r"(part_\w+\.parquet)$"))

fichier,len
str,u32
"""part_Lyon.parquet""",1009
"""part_Paris.parquet""",1011


In [62]:
# Écriture partitionnée (style Hive) : un dossier par valeur de la clé
dossier = WORKDIR / "partitionne"
ventes.with_columns(pl.col("magasin").cast(pl.String)).write_parquet(dossier, partition_by="magasin")
sorted(p.name for p in dossier.iterdir())

['magasin=Bordeaux',
 'magasin=Lille',
 'magasin=Lyon',
 'magasin=Nantes',
 'magasin=Paris']

In [63]:
pl.scan_parquet(dossier / "**/*.parquet", hive_partitioning=True) \
  .filter(pl.col("magasin") == "Lille").select(pl.len()).collect()

len
u32
963


## 10. Interface SQL

Polars sait exécuter des requêtes SQL directement sur des DataFrames / LazyFrames. Pratique pour les équipes habituées au SQL, et les deux syntaxes peuvent être mélangées.

In [64]:
ventes.sql(
    '''
    SELECT magasin,
           COUNT(*)              AS nb,
           ROUND(SUM(montant))   AS ca,
           ROUND(AVG(montant),2) AS panier_moyen
    FROM self
    WHERE categorie = 'Textile'
    GROUP BY magasin
    ORDER BY ca DESC
    '''
)

magasin,nb,ca,panier_moyen
enum,i64,f64,f64
"""Nantes""",262,217760.0,831.15
"""Bordeaux""",249,205616.0,825.77
"""Lyon""",256,198652.0,775.98
"""Paris""",240,187657.0,781.9
"""Lille""",243,180432.0,742.52


In [65]:
ctx = pl.SQLContext(ventes=ventes, magasins=info_magasins)
ctx.execute(
    '''
    SELECT v.magasin, m.directeur, ROUND(SUM(v.montant)) AS ca
    FROM ventes v
    JOIN magasins m ON CAST(v.magasin AS VARCHAR) = m.magasin
    GROUP BY v.magasin, m.directeur
    ORDER BY ca DESC
    ''',
    eager=True,
)

magasin,directeur,ca
enum,str,f64
"""Paris""","""Mme Roux""",834927.0
"""Lyon""","""M. Petit""",797448.0
"""Lille""","""Mme Diaz""",752989.0


In [66]:
# Une expression SQL utilisable dans une requête Polars classique
ventes.select("montant", pl.sql_expr("CASE WHEN montant > 1000 THEN 'gros' ELSE 'petit' END AS taille")).head(3)

montant,taille
f64,str
457.54,"""petit"""
206.43,"""petit"""
250.95,"""petit"""


## 11. Interopérabilité : pandas, NumPy, Arrow

In [67]:
import numpy as np

arr = ventes.select("quantite", "montant").to_numpy()
print(type(arr), arr.shape)

print(pl.from_numpy(np.arange(6).reshape(3, 2), schema=["a", "b"]))

<class 'numpy.ndarray'> (5000, 2)
shape: (3, 2)
┌─────┬─────┐
│ a   ┆ b   │
│ --- ┆ --- │
│ i64 ┆ i64 │
╞═════╪═════╡
│ 0   ┆ 1   │
│ 2   ┆ 3   │
│ 4   ┆ 5   │
└─────┴─────┘


In [68]:
try:
    import pandas as pd

    pdf = ventes.head(3).to_pandas()
    print(type(pdf))
    retour = pl.from_pandas(pdf)
    print(retour.schema)
except ImportError:
    print("pandas n'est pas installé (pip install pandas)")

<class 'pandas.DataFrame'>
Schema({'id_vente': Int64, 'date': Datetime(time_unit='us', time_zone=None), 'magasin': Categorical, 'categorie': Categorical, 'quantite': Int64, 'prix_unitaire': Float64, 'montant': Float64})


In [69]:
try:
    import pyarrow as pa

    table = ventes.head(3).to_arrow()          # conversion sans copie dans la plupart des cas
    print(type(table))
    print(pl.from_arrow(table).shape)
except ImportError:
    print("pyarrow n'est pas installé (pip install pyarrow)")

<class 'pyarrow.lib.Table'>
(3, 7)


**Correspondance pandas → Polars**

| pandas | Polars |
|---|---|
| `df["col"]`, `df[["a", "b"]]` | `df.select("col")`, `df.select("a", "b")` |
| `df[df.x > 0]` | `df.filter(pl.col("x") > 0)` |
| `df.assign(y=df.x * 2)` | `df.with_columns(y=pl.col("x") * 2)` |
| `df.groupby("k").agg({"x": "sum"})` | `df.group_by("k").agg(pl.col("x").sum())` |
| `df.groupby("k")["x"].transform("mean")` | `pl.col("x").mean().over("k")` |
| `df.merge(o, on="k", how="left")` | `df.join(o, on="k", how="left")` |
| `pd.concat([a, b])` | `pl.concat([a, b])` |
| `df.melt(...)` | `df.unpivot(...)` |
| `df.pivot_table(...)` | `df.pivot(...)` |
| `df.sort_values("x")` | `df.sort("x")` |
| `df.rename(columns={...})` | `df.rename({...})` |
| `df.apply(f, axis=1)` | expressions natives (ou `map_elements` en dernier recours) |
| `df.resample("1M").sum()` | `df.group_by_dynamic("date", every="1mo").agg(...)` |
| index / `loc` / `iloc` | pas d'index : `filter`, `row`, `slice`, `with_row_index` |

## 12. Nouveautés et migration vers Polars 2.0

### 12.1 Ce qui change

| Thème | Polars 2.0 |
|---|---|
| **Moteur par défaut** | `LazyFrame.collect()` utilise `engine="auto"` → **streaming** par défaut ; repli automatique sur le moteur en mémoire. Réglable via `pl.Config.set_engine_affinity(...)` ou `POLARS_ENGINE_AFFINITY`. |
| **Ordre des lignes** | Non garanti pour `join`, `group_by`, `unpivot` exécutés en streaming → utiliser `maintain_order` ou `sort`. |
| **Nouveaux types** | `Map(key, value)`, `Float16` (en plus de `Int128` / `UInt128`). |
| **Nouvelles API** | `pl.union` (concaténation sans ordre), `collect(background=True)` (instable), `pl.defer`, `pl.row_index`, `LazyFrame.pivot(..., on_columns=...)`. |
| **Interchange Protocol** | `pl.from_dataframe` n'utilise plus le *DataFrame Interchange Protocol* : seule l'interface *Arrow PyCapsule* est prise en charge, et le paramètre `allow_copy` est supprimé. |
| **API supprimées** | Les anciennes API dépréciées en 1.x ont été retirées (voir tableau ci-dessous). |
| **Nouvelles dépréciations** | `Expr.cut` → `Expr.bin_intervals`, `Expr.qcut` → `Expr.bin_quantiles`. |

> Référence officielle : le guide de migration sur <https://docs.pola.rs/releases/upgrade/>. Conseil des mainteneurs : passez d'abord à la **dernière version 1.x**, corrigez tous les `DeprecationWarning`, puis passez en 2.0.

### 12.2 Table de correspondance des anciennes API

In [70]:
anciennes_api = [
    # (objet, ancien nom, nouveau nom)
    (pl.DataFrame, "groupby", "group_by"),
    (pl.DataFrame, "melt", "unpivot"),
    (pl.DataFrame, "with_row_count", "with_row_index"),
    (pl.DataFrame, "apply", "map_rows"),
    (pl.Expr, "apply", "map_elements"),
    (pl.Expr, "map", "map_batches"),
    (pl.Expr, "cumsum", "cum_sum"),
    (pl.Expr, "cumcount", "cum_count"),
    (pl.Expr, "take", "gather"),
    (pl.Expr, "is_not", "not_"),
    (pl.Expr, "cut", "bin_intervals"),      # encore présent mais déprécié en 2.0
    (pl.Expr, "qcut", "bin_quantiles"),     # encore présent mais déprécié en 2.0
]

with pl.Config(tbl_rows=-1):   # afficher toutes les lignes
  display(pl.DataFrame(
    [
        {
            "objet": obj.__name__,
            "ancien": ancien,
            "ancien_existe_en_2.0": hasattr(obj, ancien),
            "nouveau": nouveau,
            "nouveau_existe": hasattr(obj, nouveau),
        }
        for obj, ancien, nouveau in anciennes_api
    ]
  ))

objet,ancien,ancien_existe_en_2.0,nouveau,nouveau_existe
str,str,bool,str,bool
"""DataFrame""","""groupby""",false,"""group_by""",true
"""DataFrame""","""melt""",false,"""unpivot""",true
"""DataFrame""","""with_row_count""",false,"""with_row_index""",true
"""DataFrame""","""apply""",false,"""map_rows""",true
"""Expr""","""apply""",false,"""map_elements""",true
"""Expr""","""map""",true,"""map_batches""",true
"""Expr""","""cumsum""",false,"""cum_sum""",true
"""Expr""","""cumcount""",false,"""cum_count""",true
"""Expr""","""take""",false,"""gather""",true


### 12.3 Exemple de migration

```python
# ---- Polars 1.x (ancien code) --------------------------------------------
resultat = (
    pl.scan_csv("ventes.csv")
    .join(magasins.lazy(), on="magasin")            # supposait l'ordre de gauche conservé
    .melt(id_vars="magasin", value_vars=["q1", "q2"])
    .with_row_count("n")
    .collect()                                       # moteur en mémoire par défaut
)
premieres_lignes = resultat.head(5)                 # dépendait implicitement de l'ordre

# ---- Polars 2.0 ----------------------------------------------------------
resultat = (
    pl.scan_csv("ventes.csv")
    .join(magasins.lazy(), on="magasin", maintain_order="left")
    .unpivot(index="magasin", on=["q1", "q2"])
    .with_row_index("n")
    .collect()                                       # streaming par défaut
)
premieres_lignes = resultat.sort("magasin", "variable").head(5)  # ordre explicite
```

**Liste de contrôle de migration :**

1. Mettre à jour vers la dernière 1.x et lancer la suite de tests avec `-W error::DeprecationWarning`.
2. Remplacer les API renommées (tableau ci-dessus).
3. Rechercher toutes les jointures / `group_by` / `unpivot` dont le résultat est utilisé de manière **sensible à l'ordre** (`head`, `first`, `row(0)`, comparaisons) → ajouter `maintain_order` ou `sort`.
4. Si besoin d'un comportement identique à la 1.x le temps de migrer : `pl.Config.set_engine_affinity("in-memory")`.
5. Vérifier les conversions de DataFrames tiers passant par `pl.from_dataframe` (désormais via PyCapsule Arrow uniquement).

## 13. Bonnes pratiques

1. **Pensez en expressions**, pas en boucles : évitez `for` sur les lignes et `map_elements`.
2. **Utilisez le mode lazy** (`scan_*` + `collect` / `sink_*`) pour tout traitement de fichiers.
3. **Typez au plus juste** : `Enum`/`Categorical` pour les chaînes répétitives, `Int32`/`Float32` si la précision suffit, `schema_overrides` à la lecture.
4. **Filtrez et sélectionnez tôt** (l'optimiseur le fait en lazy, mais pas en eager).
5. **Ne supposez jamais un ordre implicite** : triez explicitement, ou utilisez `maintain_order`.
6. **Chaînez les opérations** entre parenthèses pour un code lisible ; utilisez `.pipe(fonction)` pour factoriser des étapes.
7. **Inspectez** avec `explain()`, `collect_schema()`, `glimpse()`, et `POLARS_VERBOSE=1` pour comprendre les replis de moteur.
8. **Préférez Parquet** au CSV pour le stockage intermédiaire.

In [71]:
# Exemple de pipeline factorisé avec pipe
def ajouter_calendrier(lf: pl.LazyFrame) -> pl.LazyFrame:
    return lf.with_columns(
        annee=pl.col("date").dt.year(),
        mois=pl.col("date").dt.month(),
        jour_semaine=pl.col("date").dt.weekday(),
    )


def ca_par(lf: pl.LazyFrame, *cles: str) -> pl.LazyFrame:
    return lf.group_by(*cles).agg(ca=pl.col("montant").sum().round(0)).sort(*cles)


pl.scan_parquet(fichier).pipe(ajouter_calendrier).pipe(ca_par, "jour_semaine").collect()

jour_semaine,ca
i8,f64
1,570705.0
2,562968.0
3,589443.0
4,591044.0
5,566946.0
6,575497.0
7,603088.0


## 14. Exercices corrigés

Essayez de résoudre chaque exercice dans la cellule vide avant de regarder la solution. Tous les exercices utilisent le DataFrame `ventes`.

### Exercice 1 — Filtrage et sélection
Afficher les 5 plus grosses ventes du magasin de **Nantes** en catégorie **Maison**, avec les colonnes `date`, `quantite`, `montant`.

In [72]:
# Votre solution ici

In [73]:
# Solution
(
    ventes.filter(pl.col("magasin") == "Nantes", pl.col("categorie") == "Maison")
    .top_k(5, by="montant")
    .select("date", "quantite", "montant")
)

date,quantite,montant
datetime[μs],i64,f64
2025-01-06 10:00:00,10,2962.5
2025-12-13 05:00:00,10,2925.4
2025-10-23 11:00:00,10,2865.7
2025-12-21 03:00:00,10,2759.0
2025-02-21 09:00:00,10,2750.4


### Exercice 2 — Agrégation
Pour chaque catégorie, calculer le chiffre d'affaires total, le nombre de ventes et la **part (%) du CA total**, triés par CA décroissant.

In [74]:
# Votre solution ici

In [75]:
# Solution
(
    ventes.group_by("categorie")
    .agg(ca=pl.col("montant").sum(), nb=pl.len())
    .with_columns(part_pct=(pl.col("ca") / pl.col("ca").sum() * 100).round(1), ca=pl.col("ca").round(0))
    .sort("ca", descending=True)
)

categorie,ca,nb,part_pct
cat,f64,u32,f64
"""Maison""",1.031962e6,1272,25.4
"""Alimentaire""",1.022183e6,1227,25.2
"""High-tech""",1.015431e6,1251,25.0
"""Textile""",990117.0,1250,24.4


### Exercice 3 — Fonction fenêtre
Ajouter à chaque vente une colonne `ecart_moyenne_cat` : l'écart entre son montant et le montant moyen de **sa catégorie dans son magasin**. Afficher les 5 ventes au plus grand écart.

In [76]:
# Votre solution ici

In [77]:
# Solution
(
    ventes.with_columns(
        ecart_moyenne_cat=(pl.col("montant") - pl.col("montant").mean().over("magasin", "categorie")).round(2)
    )
    .top_k(5, by="ecart_moyenne_cat")
    .select("magasin", "categorie", "montant", "ecart_moyenne_cat")
)

magasin,categorie,montant,ecart_moyenne_cat
enum,cat,f64,f64
"""Paris""","""Maison""",2983.5,2189.4
"""Paris""","""Textile""",2968.9,2187.0
"""Lyon""","""High-tech""",2978.6,2175.94
"""Lille""","""High-tech""",2936.7,2172.91
"""Nantes""","""Maison""",2962.5,2170.55


### Exercice 4 — Séries temporelles
Calculer le CA **hebdomadaire** de chaque magasin, puis, pour chaque magasin, la **meilleure semaine** de l'année.

In [78]:
# Votre solution ici

In [79]:
# Solution
(
    ventes.group_by_dynamic("date", every="1w", group_by="magasin")
    .agg(ca=pl.col("montant").sum().round(0))
    .filter(pl.col("ca") == pl.col("ca").max().over("magasin"))
    .sort("magasin")
)

magasin,date,ca
enum,datetime[μs],f64
"""Paris""",2025-08-04 00:00:00,26782.0
"""Lyon""",2025-03-17 00:00:00,28694.0
"""Lille""",2025-06-30 00:00:00,24354.0
"""Nantes""",2025-03-31 00:00:00,35880.0
"""Bordeaux""",2025-06-02 00:00:00,26513.0


### Exercice 5 — Lazy, jointure et ordre (spécial 2.0)
À partir de `pl.scan_parquet(fichier)` :
1. joindre `info_magasins` (en lazy) pour récupérer le directeur, en **conservant l'ordre** de la table des ventes ;
2. ne garder que les ventes de plus de 1 000 € ;
3. renvoyer, par directeur, le nombre de ces ventes, **trié** par nombre décroissant ;
4. afficher le plan optimisé et vérifier que le filtre est poussé avant la jointure.

In [80]:
# Votre solution ici

In [81]:
# Solution
requete_ex5 = (
    pl.scan_parquet(fichier)
    .with_columns(pl.col("magasin").cast(pl.String))
    .join(info_magasins.lazy(), on="magasin", how="inner", maintain_order="left")
    .filter(pl.col("montant") > 1_000)
    .group_by("directeur")
    .agg(nb_grosses_ventes=pl.len())
    .sort("nb_grosses_ventes", descending=True)
)
print(requete_ex5.explain())
requete_ex5.collect()

SORT BY [descending: [true]] [col("nb_grosses_ventes")]
  AGGREGATE[maintain_order: false]
    [len().alias("nb_grosses_ventes")] BY [col("directeur")]
    FROM
    simple π 1/1 ["directeur"]
      INNER JOIN:
      LEFT PLAN ON: [col("magasin")]
         WITH_COLUMNS:
         [col("magasin").strict_cast(String)] 
          simple π 1/1 ["magasin"]
            Parquet SCAN [/tmp/formation_polars_xxxx/ventes.parquet]
            PROJECT 2/7 COLUMNS
            SELECTION: col("montant") > 1000.0
            ESTIMATED ROWS: 5000
      RIGHT PLAN ON: [col("magasin")]
        DF ["magasin", "surface_m2", "directeur"]; PROJECT["directeur", "magasin"] 2/3 COLUMNS
      END INNER JOIN


directeur,nb_grosses_ventes
str,u32
"""Mme Roux""",339
"""M. Petit""",321
"""Mme Diaz""",290


### Exercice 6 — Migration
Le code suivant a été écrit pour une ancienne version de Polars. Réécrivez-le pour Polars 2.0.

```python
df.groupby("magasin").agg(pl.col("montant").apply(lambda s: s.sum()))
df.melt(id_vars="magasin", value_vars=["quantite", "montant"])
df.with_row_count()
df.with_columns(pl.col("montant").cumsum().alias("cumul"))
```

In [82]:
# Votre solution ici

In [83]:
# Solution
petit = ventes.head(20)

print(petit.group_by("magasin", maintain_order=True).agg(pl.col("montant").sum()))
print(petit.unpivot(index="magasin", on=["quantite", "montant"]).head(3))
print(petit.with_row_index().head(3))
print(petit.with_columns(cumul=pl.col("montant").cum_sum()).select("montant", "cumul").head(3))

shape: (5, 2)
┌──────────┬─────────┐
│ magasin  ┆ montant │
│ ---      ┆ ---     │
│ enum     ┆ f64     │
╞══════════╪═════════╡
│ Paris    ┆ 1942.84 │
│ Lyon     ┆ 9960.93 │
│ Nantes   ┆ 1426.31 │
│ Lille    ┆ 4134.48 │
│ Bordeaux ┆ 669.75  │
└──────────┴─────────┘
shape: (3, 3)
┌─────────┬──────────┬───────┐
│ magasin ┆ variable ┆ value │
│ ---     ┆ ---      ┆ ---   │
│ enum    ┆ str      ┆ f64   │
╞═════════╪══════════╪═══════╡
│ Paris   ┆ quantite ┆ 2.0   │
│ Lyon    ┆ quantite ┆ 1.0   │
│ Nantes  ┆ quantite ┆ 7.0   │
└─────────┴──────────┴───────┘
shape: (3, 8)
┌───────┬──────────┬───────────────┬─────────┬─────────────┬──────────┬───────────────┬─────────┐
│ index ┆ id_vente ┆ date          ┆ magasin ┆ categorie   ┆ quantite ┆ prix_unitaire ┆ montant │
│ ---   ┆ ---      ┆ ---           ┆ ---     ┆ ---         ┆ ---      ┆ ---           ┆ ---     │
│ u32   ┆ i64      ┆ datetime[μs]  ┆ enum    ┆ cat         ┆ i64      ┆ f64           ┆ f64     │
╞═══════╪══════════╪══════════════

---
## Pour aller plus loin

* Documentation officielle : <https://docs.pola.rs>
* Guide utilisateur : <https://docs.pola.rs/user-guide/>
* Référence de l'API Python : <https://docs.pola.rs/api/python/stable/reference/>
* Guides de migration : <https://docs.pola.rs/releases/upgrade/>
* Code source : <https://github.com/pola-rs/polars>

Félicitations, vous avez terminé le module de formation Polars 2.0 ! 🎉